# Snowball Strategy

In [ ]:
from datetime import UTC, datetime, timedelta
from html import escape
from itertools import pairwise
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from aws import AthenaDataSource
from core import (
    BacktestTaskDefinition,
    CandleGranularity,
    CsvResultStore,
    CurrencyPair,
    InMemoryTaskRegistry,
    LogLevel,
    Money,
    Pips,
    SpreadFilter,
    StrategyParameters,
    TaskManager,
    TaskProfilingConfig,
    TaskResultRecorder,
    TickGranularity,
    TickGranularityFilter,
    TqdmProgressReporter,
    configure_logging,
)
from IPython.display import HTML, display
from matplotlib.patches import Rectangle
from snowball import SnowballStrategy

configure_logging(level=LogLevel.WARNING)


## Parameters

In [ ]:
INSTRUMENT = CurrencyPair.of("USD_JPY")
START_AT = datetime(2026, 1, 1, tzinfo=UTC)
END_AT = datetime(2026, 1, 31, 23, 59, 59, 999999, tzinfo=UTC)
SAMPLE_GRANULARITY = TickGranularity.TICK
CHART_DAY_AGGS_THRESHOLD_DAYS = 31
PROFILE_DIR = Path("profiles") / "snowball"
RESULT_DIR = Path("results") / "snowball"
LOCAL_TIMEZONE = datetime.now().astimezone().tzinfo

data_source = AthenaDataSource.from_env().with_filters(
    SpreadFilter.of(Pips("5")),
    TickGranularityFilter.of(SAMPLE_GRANULARITY),
)

parameters = SnowballStrategy.normalize_parameters(
    StrategyParameters.of(
        sizing={
            # choices: "fixed", "balance_based"
            "mode": "balance_based",
            "base_units": "1500",
            "balance_based": {
                "reference_balance": {
                    "amount": "3000000",
                    "currency": "JPY",
                },
                "reference_units": "1500",
                "round_step_units": "1",
                "min_units": "1",
            },
        },
        grid={
            "max_retracements_per_layer": 6,
            "max_layers": 30,
            "refill": {
                "enabled": True,
            },
        },
        cycle={
            "hedging_enabled": True,
            "reseed_when_all_positions_pending_rebuild": True,
        },
        forward={
            "take_profit_pips": "30",
        },
        counter={
            "interval": {
                # choices: "constant", "additive", "subtractive"
                #          "multiplicative", "divisive", "manual"
                "mode": "divisive",
                "head_pips": "30",
                "tail_pips": "10",
                "gamma": "1.3",
            },
            "take_profit": {"mode": "weighted_avg"},
        },
        stop_loss={
            "enabled": False,
            # choices: "auto", "distance"
            "mode": "auto",
        },
        rebuild={
            "enabled": True,
            "price": {
                # choices: "original_entry_price", "stop_loss_exit_price"
                "entry_price_mode": "original_entry_price",
            },
            "stop_loss": {
                # choices: "same_price", "same_distance", "manual_distance"
                "mode": "same_distance",
            },
            "take_profit": {
                # choices: "same_price", "same_distance", "progressive_distance"
                "mode": "same_distance"
            },
        },
        protection={
            "shrink_enabled": False,
            "shrink_start_margin_percent": "70",
            "shrink_target_margin_percent": "50",
            "emergency_enabled": True,
            "emergency_margin_percent": "95",
        },
        account={
            "margin_rate": "0.04",
            "quote_to_account_rate": "1",
        },
    )
)


## Executuin

In [ ]:
definition = BacktestTaskDefinition(
    name=f"{INSTRUMENT} Snowball backtest",
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    initial_balance=Money.of("3000000", "JPY"),
    parameters=parameters,
)

strategy = SnowballStrategy(
    name="snowball",
    parameters=definition.parameters,
)

recorder = TaskResultRecorder(
    stores=(CsvResultStore(RESULT_DIR),),
    metric_interval=timedelta(minutes=5),
)

with TaskManager(
    registry=InMemoryTaskRegistry(),
    observers=(recorder,),
    event_handlers=(recorder,),
    profiling=TaskProfilingConfig.for_backtest_period(
        start_at=START_AT,
        end_at=END_AT,
        cprofile=False,
        cprofile_output_path=PROFILE_DIR,
        pyinstrument=False,
        pyinstrument_output_path=PROFILE_DIR,
    ),
) as manager:
    run = manager.start_backtest(definition, data_source=data_source, strategy=strategy)
    final_task = run.wait(progress=TqdmProgressReporter())

print(f"Events recorded: {len(recorder.event_records(final_task.id))}")
print(f"Trades recorded: {len(recorder.trade_summaries(final_task.id))}")


## Results

In [ ]:
events = recorder.event_records(final_task.id)
trades = recorder.trade_summaries(final_task.id)
cycles = recorder.cycle_summaries(final_task.id)
task_summary = recorder.task_summary(final_task)
metrics = tuple(metric for metric in recorder.memory.metrics if metric.task_id == final_task.id)


def local_datetime(value):
    return value.astimezone(LOCAL_TIMEZONE) if value is not None else None


def text(value):
    return str(value) if value is not None else None


def metadata_text(value):
    return str(value.to_jsonable()) if value else None


def display_paginated_df(df, *, title, table_id, page_size=100):
    total_rows = len(df)
    if total_rows == 0:
        display(HTML(f"""
        <section class="paged-frame" id="{table_id}">
            <h3>{escape(title)}</h3>
            <p>0 rows</p>
        </section>
        """))
        return

    table_html = df.to_html(
        index=False,
        escape=True,
        max_rows=None,
        max_cols=None,
        border=0,
        classes="paged-table",
    )
    display(HTML(f"""
    <section class="paged-frame" id="{table_id}">
        <style>
            #{table_id} {{ margin: 16px 0 28px; }}
            #{table_id} h3 {{ margin: 0 0 8px; font-size: 16px; }}
            #{table_id} .paged-toolbar {{
                align-items: center;
                display: flex;
                gap: 8px;
                margin-bottom: 8px;
            }}
            #{table_id} .paged-toolbar button {{
                border: 1px solid #cbd5e1;
                border-radius: 4px;
                background: #f8fafc;
                cursor: pointer;
                padding: 4px 10px;
            }}
            #{table_id} .paged-toolbar button:disabled {{
                color: #94a3b8;
                cursor: default;
            }}
            #{table_id} .paged-scroll {{ overflow-x: auto; }}
            #{table_id} table.paged-table {{
                border-collapse: collapse;
                font-size: 12px;
                white-space: nowrap;
                width: max-content;
            }}
            #{table_id} table.paged-table th,
            #{table_id} table.paged-table td {{
                border: 1px solid #e2e8f0;
                padding: 4px 8px;
                text-align: left;
                vertical-align: top;
            }}
            #{table_id} table.paged-table th {{ background: #f8fafc; }}
        </style>
        <h3>{escape(title)}</h3>
        <div class="paged-toolbar">
            <button type="button" data-prev>Prev</button>
            <span data-page-label></span>
            <button type="button" data-next>Next</button>
        </div>
        <div class="paged-scroll">{table_html}</div>
        <script>
        (() => {{
            const root = document.getElementById("{table_id}");
            if (!root) return;
            const rows = Array.from(root.querySelectorAll("tbody tr"));
            const pageSize = {page_size};
            const totalPages = Math.max(1, Math.ceil(rows.length / pageSize));
            let currentPage = 0;
            const label = root.querySelector("[data-page-label]");
            const prev = root.querySelector("[data-prev]");
            const next = root.querySelector("[data-next]");
            const render = () => {{
                const start = currentPage * pageSize;
                const end = start + pageSize;
                rows.forEach((row, index) => {{
                    row.style.display = index >= start && index < end ? "" : "none";
                }});
                label.textContent = `Page ${{currentPage + 1}} / ${{totalPages}} (${{rows.length}} rows)`;
                prev.disabled = currentPage === 0;
                next.disabled = currentPage >= totalPages - 1;
            }};
            prev.addEventListener("click", () => {{
                if (currentPage > 0) {{
                    currentPage -= 1;
                    render();
                }}
            }});
            next.addEventListener("click", () => {{
                if (currentPage < totalPages - 1) {{
                    currentPage += 1;
                    render();
                }}
            }});
            render();
        }})();
        </script>
    </section>
    """))


signals_df = pd.DataFrame(
    {
        "timestamp": local_datetime(event.timestamp),
        "action": event.action.value,
        "display_id": event.display_id,
        "units": text(event.units),
        "price": text(event.price),
        "rule": event.rule,
        "cycle_id": event.cycle_id,
        "direction": event.direction,
        "entry_role": event.entry_role,
        "layer_number": event.layer_number,
        "slot_number": event.slot_number,
        "build_number": event.build_number,
        "close_reason": event.close_reason,
        "is_rebuild": event.is_rebuild,
        "planned_entry_price": text(event.planned_entry_price),
        "filled_entry_price": text(event.filled_entry_price),
        "planned_take_profit_price": text(event.planned_take_profit_price),
        "filled_take_profit_price": text(event.filled_take_profit_price),
        "planned_stop_loss_price": text(event.planned_stop_loss_price),
        "filled_stop_loss_price": text(event.filled_stop_loss_price),
        "planned_rebuild_price": text(event.planned_rebuild_price),
        "filled_rebuild_price": text(event.filled_rebuild_price),
        "realized_pl": text(event.realized_pl),
    }
    for event in events
)

trade_summary_df = pd.DataFrame(
    {
        "task_id": trade.task_id,
        "trade_id": trade.trade_id,
        "instrument": trade.instrument,
        "side": None if trade.side is None else trade.side.value,
        "direction": trade.direction,
        "display_id": trade.display_id,
        "cycle_id": trade.cycle_id,
        "entry_role": trade.entry_role,
        "layer_number": trade.layer_number,
        "slot_number": trade.slot_number,
        "build_number": trade.build_number,
        "opened_at": local_datetime(trade.opened_at),
        "closed_at": local_datetime(trade.closed_at),
        "close_reason": trade.close_reason,
        "units": text(trade.units),
        "planned_entry_price": text(trade.planned_entry_price),
        "filled_entry_price": text(trade.filled_entry_price),
        "planned_take_profit_price": text(trade.planned_take_profit_price),
        "filled_take_profit_price": text(trade.filled_take_profit_price),
        "planned_stop_loss_price": text(trade.planned_stop_loss_price),
        "filled_stop_loss_price": text(trade.filled_stop_loss_price),
        "planned_rebuild_price": text(trade.planned_rebuild_price),
        "filled_rebuild_price": text(trade.filled_rebuild_price),
        "realized_pl": text(trade.realized_pl),
    }
    for trade in trades
)

cycle_summary_df = pd.DataFrame(
    {
        "task_id": cycle.task_id,
        "cycle_id": cycle.cycle_id,
        "instrument": cycle.instrument,
        "trade_ids": list(cycle.trade_ids),
        "opened_at": local_datetime(cycle.opened_at),
        "closed_at": local_datetime(cycle.closed_at),
        "trade_count": cycle.trade_count,
        "open_trade_count": cycle.open_trade_count,
        "closed_trade_count": cycle.closed_trade_count,
        "realized_pl": text(cycle.realized_pl),
    }
    for cycle in cycles
)

task_summary_df = pd.DataFrame(
    []
    if task_summary is None
    else [
        {
            "task_id": task_summary.task_id,
            "instrument": task_summary.instrument,
            "task_name": task_summary.task_name,
            "status": task_summary.status,
            "started_at": local_datetime(task_summary.started_at),
            "finished_at": local_datetime(task_summary.finished_at),
            "trade_count": task_summary.trade_count,
            "open_trade_count": task_summary.open_trade_count,
            "closed_trade_count": task_summary.closed_trade_count,
            "realized_pl": text(task_summary.realized_pl),
        }
    ]
)

metrics_df = pd.DataFrame(
    {
        "timestamp": local_datetime(metric.timestamp),
        "realized_pl": float(metric.realized_pl.amount),
        "unrealized_pl": float(metric.unrealized_pl.amount),
        "total_pl": float(metric.total_pl.amount),
        "currency": metric.total_pl.currency.code,
        "open_trade_count": metric.open_trade_count,
        "closed_trade_count": metric.closed_trade_count,
    }
    for metric in metrics
)

events_df = signals_df

display_paginated_df(task_summary_df, title="Task Summary", table_id="task-summary", page_size=100)
display_paginated_df(cycle_summary_df, title="Cycle Summary", table_id="cycle-summary", page_size=100)
display_paginated_df(trade_summary_df, title="Trade Summary", table_id="trade-summary", page_size=100)

## Chart


In [ ]:
chart_granularity = (
    CandleGranularity.DAY
    if END_AT - START_AT > timedelta(days=CHART_DAY_AGGS_THRESHOLD_DAYS)
    else CandleGranularity.MINUTE_1
)

candles = tuple(
    data_source.candles(
        instrument=INSTRUMENT,
        granularity=chart_granularity,
        start_at=START_AT,
        end_at=END_AT,
    )
)

candles_df = pd.DataFrame(
    {
        "timestamp": local_datetime(candle.timestamp),
        "open": float(candle.open.amount),
        "high": float(candle.high.amount),
        "low": float(candle.low.amount),
        "close": float(candle.close.amount),
        "volume": candle.volume,
    }
    for candle in candles
)

fig, (price_ax, metric_ax) = plt.subplots(
    2,
    1,
    figsize=(16, 10),
    sharex=True,
    gridspec_kw={"height_ratios": (3, 1), "hspace": 0.08},
)

if not candles_df.empty:
    x_values = mdates.date2num(candles_df["timestamp"])
    positive_deltas = [right - left for left, right in pairwise(x_values) if right > left]
    width = min(positive_deltas) * 0.8 if positive_deltas else 0.7
    if chart_granularity == CandleGranularity.MINUTE_1:
        width = min(width, 1 / 24 / 60 * 0.8)

    for x, row in zip(x_values, candles_df.itertuples(index=False), strict=False):
        color = "#16833a" if row.close >= row.open else "#b42318"
        price_ax.vlines(x, row.low, row.high, color=color, linewidth=0.8, alpha=0.9)
        lower = min(row.open, row.close)
        height = abs(row.close - row.open)
        if height == 0:
            price_ax.hlines(row.open, x - width / 2, x + width / 2, color=color, linewidth=1.2)
        else:
            price_ax.add_patch(
                Rectangle(
                    (x - width / 2, lower),
                    width,
                    height,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=0.6,
                    alpha=0.75,
                )
            )


def event_points(predicate):
    return [
        (local_datetime(event.timestamp), float(event.price.amount))
        for event in events
        if event.price is not None and predicate(event)
    ]


def scatter(points, *, label, marker, color):
    if not points:
        return
    timestamps, prices = zip(*points, strict=False)
    price_ax.scatter(timestamps, prices, label=label, marker=marker, color=color, s=48, zorder=5)


scatter(
    event_points(lambda event: event.action.value == "open_trade" and not event.is_rebuild),
    label="open",
    marker="^",
    color="#2563eb",
)
scatter(
    event_points(
        lambda event: event.action.value == "close_trade" and event.close_reason != "stop_loss"
    ),
    label="close",
    marker="v",
    color="#0f766e",
)
scatter(
    event_points(lambda event: event.action.value == "open_trade" and event.is_rebuild),
    label="rebuild",
    marker="D",
    color="#9333ea",
)
scatter(
    event_points(lambda event: event.close_reason == "stop_loss"),
    label="stop_loss",
    marker="x",
    color="#dc2626",
)

price_ax.set_title(f"{INSTRUMENT} {chart_granularity.value} Snowball")
price_ax.set_ylabel(str(INSTRUMENT.quote))
price_ax.grid(True, axis="y", alpha=0.25)
handles, labels = price_ax.get_legend_handles_labels()
if handles:
    price_ax.legend(handles, labels, loc="best")

if metrics_df.empty:
    metric_ax.text(0.5, 0.5, "No metrics", ha="center", va="center", transform=metric_ax.transAxes)
else:
    metric_ax.plot(
        metrics_df["timestamp"],
        metrics_df["realized_pl"],
        label="realized_pl",
        color="#2563eb",
        linewidth=1.4,
    )
    metric_ax.plot(
        metrics_df["timestamp"],
        metrics_df["unrealized_pl"],
        label="unrealized_pl",
        color="#f97316",
        linewidth=1.4,
    )
    metric_ax.plot(
        metrics_df["timestamp"],
        metrics_df["total_pl"],
        label="total_pl",
        color="#111827",
        linewidth=1.6,
    )
    currency = metrics_df["currency"].dropna().iloc[0] if metrics_df["currency"].notna().any() else ""
    metric_ax.set_ylabel(f"P/L {currency}".strip())
    metric_ax.legend(loc="best", ncols=3)

metric_ax.grid(True, axis="y", alpha=0.25)
metric_ax.xaxis_date()
metric_ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M", tz=LOCAL_TIMEZONE))
fig.autofmt_xdate()
plt.show()


In [ ]:
profile = run.profile()
profile_outputs = {
    "cprofile": profile.cprofile_output_path,
    "pyinstrument": profile.pyinstrument_output_path,
}
if profile.cprofile_output_path is not None:
    profile_outputs["snakeviz"] = f"uv run snakeviz {profile.cprofile_output_path}"
    profile_outputs["tuna"] = f"uv run tuna {profile.cprofile_output_path}"
if profile.pyinstrument_output_path is not None:
    profile_outputs["html"] = f"open {profile.pyinstrument_output_path}"
profile_outputs